# Proyecto Integrador 2 · Parte I — Base relacional en Spark
## Tareas 1 a 5 y 7: resolución de proveedores, matriz entidad×proveedor, vectores de características y particionamiento

**Qué hace este notebook**
1. Lee los Parquet de la limpieza del Proyecto 1 (no los vuelve a limpiar).
2. **Resuelve proveedores:** decide qué es un mismo proveedor y qué se deja fuera, y le da un número (`proveedor_idx`).
3. Construye la **matriz entidad×proveedor** y la separa en *train* y *test* por fecha.
4. Calcula los **vectores de características** de proveedores y de entidades.
5. Explica el **particionamiento** y cómo cambia el plan de ejecución.

**Por qué Spark:** la Parte I lo exige, y aquí se une y agrega la tabla de 2.5 millones de postores con las adjudicaciones.
**Entrada:** Parquet del Proyecto 1. **Salida:** 6 Parquet en `RUTA_P2` (los leen los notebooks 2 a 6) y `reporte_base.json` para la página web.

**Preguntas de interés que prepara (ver README):**
- **P1** (tipos de proveedores y casos raros): aquí se construyen los vectores que se agrupan en el notebook 3.
- **P2** (sustitutos y afinidades): aquí se construye la matriz que usan los recomendadores de los notebooks 4 a 6.

## 0. Configuración

In [1]:
import os, json, time
from io import StringIO
from contextlib import redirect_stdout
import pandas as pd

USAR_DRIVE = False   # True si corres en Colab con Drive
RUTA_PARQUET = os.environ.get("P2_PARQUET", "/content/drive/MyDrive/proyecto_integrador_seace/parquet")                    # Parquet del Proyecto 1
RUTA_P2      = os.environ.get("P2_DATOS",   "/content/drive/MyDrive/proyecto_integrador_seace/p2")                         # Parquet intermedios de esta parte
WEB          = os.environ.get("P2_WEB",     "/content/drive/MyDrive/proyecto_integrador_seace/web_artefactos/p2")          # archivos para la página web

FECHA_CORTE   = "2026-03-01"   # procesos publicados antes = train, desde esta fecha = test (los últimos 6 meses)
MIN_ENT_PROV  = 2              # un proveedor entra a la matriz si tiene al menos 2 entidades distintas en train
MIN_PROV_ENT  = 3              # una entidad entra a la matriz si tiene al menos 3 proveedores distintos en train

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(RUTA_P2, exist_ok=True)
os.makedirs(WEB, exist_ok=True)

def guardar_json(nombre, obj):
    with open(f"{WEB}/{nombre}", "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print("guardado:", nombre)

## 1. Spark y lectura

In [2]:
# Solo Colab: !pip install -q pyspark==3.5.3
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder.master("local[*]").appName("p2_base")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())

procesos = spark.read.parquet(f"{RUTA_PARQUET}/procesos").select(
    "release_id", "entidad_id", "fecha_pub", "categoria", "n_postores")
adj  = spark.read.parquet(f"{RUTA_PARQUET}/adjudicaciones").select(
    "release_id", "proveedor_ruc", "proveedor", "proveedor_extranjero", "monto_pen")
post = spark.read.parquet(f"{RUTA_PARQUET}/postores").select("release_id", "ruc", "postor")
ent  = spark.read.parquet(f"{RUTA_PARQUET}/entidades")

print("procesos:", procesos.count(), "| adjudicaciones:", adj.count(), "| postores:", post.count(), "| entidades:", ent.count())

procesos: 209740 | adjudicaciones: 174908 | postores: 2493612 | entidades: 2992


## 2. Tareas 1 y 2 — Resolución de proveedores
**El problema:** si un mismo proveedor aparece con dos identificadores, en el grafo y en la matriz se vuelve **dos nodos**. Primero miramos qué identificadores hay.

Un RUC peruano de empresa o persona tiene 11 dígitos. Veamos los largos que aparecen en las adjudicaciones.

In [3]:
largos = (adj.filter(F.col("proveedor_ruc").isNotNull())
          .withColumn("largo", F.length("proveedor_ruc"))
          .groupBy("largo").agg(F.count("*").alias("adjudicaciones"), F.countDistinct("proveedor_ruc").alias("ids_distintos"))
          .orderBy("largo"))
largos.show()

print("Los 5 nombres con más identificadores distintos entre los de 7 dígitos:")
(adj.filter(F.length("proveedor_ruc") == 7)
 .groupBy("proveedor").agg(F.countDistinct("proveedor_ruc").alias("ids_distintos"))
 .orderBy(F.desc("ids_distintos")).show(5, truncate=False))

+-----+--------------+-------------+
|largo|adjudicaciones|ids_distintos|
+-----+--------------+-------------+
|    6|            12|           12|
|    7|         38084|        38038|
|    9|             1|            1|
|   11|        133576|        39219|
+-----+--------------+-------------+

Los 5 nombres con más identificadores distintos entre los de 7 dígitos:


+---------------------+-------------+
|proveedor            |ids_distintos|
+---------------------+-------------+
|CONSORCIO SANTA ROSA |193          |
|CONSORCIO SAN JUAN   |122          |
|CONSORCIO SAN ANTONIO|114          |
|CONSORCIO SAN MARTIN |105          |
|CONSORCIO SAN PEDRO  |100          |
+---------------------+-------------+
only showing top 5 rows



### Lectura y decisión
- Los **identificadores de 7 dígitos** son consorcios (y otras asociaciones temporales). **No son un RUC real:** cada consorcio recibe un número propio en cada proceso, y el mismo nombre aparece con muchos números (por ejemplo, "CONSORCIO SANTA ROSA" con decenas de ids).
- **Decisión:** los marcamos con `es_consorcio` y los **dejamos fuera** de la matriz, el grafo y el clustering. **No los unimos por nombre**, porque uniría consorcios distintos que solo se llaman igual.
- Los RUC de 11 dígitos son estables: casi todos tienen un solo nombre. Si tienen varias variantes, nos quedamos con el **nombre más frecuente**.
- Las adjudicaciones **sin RUC** también se dejan fuera. Todo queda contado en `reporte_base.json`.
- **MinHash/LSH del Proyecto 1 no hace falta aquí**: el RUC ya identifica al proveedor y los nombres casi iguales con RUC distinto son consorcios.

In [4]:
def solo_ruc(df, ruc, nombre):
    # observaciones (RUC, nombre) con RUC real: no nulo y que no sea de 7 dígitos
    return (df.select(F.col(ruc).alias("ruc"), F.upper(F.trim(F.col(nombre))).alias("nombre"))
              .filter(F.col("ruc").isNotNull() & (F.length("ruc") != 7)))

obs = solo_ruc(adj, "proveedor_ruc", "proveedor").unionByName(solo_ruc(post, "ruc", "postor"))

# nombre más frecuente de cada RUC y cuántos nombres distintos tuvo
cuenta = obs.groupBy("ruc", "nombre").count()
w_orden = Window.partitionBy("ruc").orderBy(F.desc("count"), "nombre")
w_total = Window.partitionBy("ruc")
nombres = (cuenta.withColumn("pos", F.row_number().over(w_orden))
                 .withColumn("n_variantes", F.count("*").over(w_total))
                 .filter("pos = 1").select("ruc", "nombre", "n_variantes"))

extranjero = adj.filter(F.col("proveedor_ruc").isNotNull()).groupBy(F.col("proveedor_ruc").alias("ruc")) \
                .agg(F.max(F.col("proveedor_extranjero").cast("int")).alias("extranjero"))

proveedor_dim = (nombres.join(extranjero, "ruc", "left")
                 .withColumn("extranjero", F.coalesce(F.col("extranjero"), F.lit(0)).cast("boolean"))
                 .withColumn("proveedor_idx", F.row_number().over(Window.orderBy("ruc")) - 1)   # 0, 1, 2, ...
                 .select("proveedor_idx", "ruc", "nombre", "n_variantes", "extranjero")
                 .cache())
n_prov = proveedor_dim.count()
print("proveedores con RUC real (en adjudicaciones o postores):", n_prov)
print("RUC con más de un nombre:", proveedor_dim.filter("n_variantes > 1").count())

entidad_dim = (ent.select("entidad_id", F.col("entidad").alias("nombre"), "entidad_tipo", "entidad_departamento")
               .withColumn("entidad_idx", F.row_number().over(Window.orderBy("entidad_id")) - 1)
               .select("entidad_idx", "entidad_id", "nombre", "entidad_tipo", "entidad_departamento").cache())
print("entidades:", entidad_dim.count())

proveedores con RUC real (en adjudicaciones o postores): 104618


RUC con más de un nombre: 630


entidades: 2992


In [5]:
# Embudo de adjudicaciones: cuántas se quedan y cuántas se dejan fuera
n_adj = adj.count()
sin_ruc = adj.filter(F.col("proveedor_ruc").isNull()).count()
consorcio = adj.filter(F.length("proveedor_ruc") == 7).count()
usables = n_adj - sin_ruc - consorcio
embudo = {"adjudicaciones_total": n_adj, "sin_ruc": sin_ruc, "consorcio_7_digitos": consorcio, "usables_con_ruc_real": usables}
print(embudo)
print(f"se deja fuera {100*(sin_ruc+consorcio)/n_adj:.1f}% de las adjudicaciones")

n_post = post.count()
post_consorcio = post.filter(F.length("ruc") == 7).count()
print(f"postores: {n_post:,} filas; {post_consorcio:,} ({100*post_consorcio/n_post:.1f}%) son ids de consorcio")

{'adjudicaciones_total': 174908, 'sin_ruc': 3235, 'consorcio_7_digitos': 38084, 'usables_con_ruc_real': 133589}
se deja fuera 23.6% de las adjudicaciones


postores: 2,493,612 filas; 121,999 (4.9%) son ids de consorcio


## 3. Tarea 3 — Matriz entidad×proveedor y separación train/test
**Interacción:** una entidad *adjudicó* a un proveedor. Usamos **adjudicaciones** y no contratos, porque hay adjudicaciones sin contrato vinculado (el enunciado pide decidir esto): así no perdemos señal.

Para cada par (entidad, proveedor) guardamos:
- `n_adjudicaciones`: cuántas veces la entidad adjudicó a ese proveedor.
- `monto_total`: suma de los montos en soles.

**Separación por fecha:** los procesos publicados **antes de `FECHA_CORTE`** son *train* y los de **desde esa fecha** son *test*. Ningún método ve el test al entrenar.

**Universo:** para recomendar se necesita algo de historia. Entran a la matriz los proveedores con al menos `MIN_ENT_PROV` entidades y las entidades con al menos `MIN_PROV_ENT` proveedores **en train**. Los demás tienen tan pocos datos que no se les puede recomendar ni evaluar.

In [6]:
a = (adj.filter(F.col("proveedor_ruc").isNotNull() & (F.length("proveedor_ruc") != 7))
        .join(procesos.select("release_id", "entidad_id", "fecha_pub"), "release_id")
        .join(proveedor_dim.select("proveedor_idx", F.col("ruc").alias("proveedor_ruc")), "proveedor_ruc")
        .join(entidad_dim.select("entidad_idx", "entidad_id"), "entidad_id")
        .select("entidad_idx", "entidad_id", "proveedor_idx", "release_id", "fecha_pub", "monto_pen")
        .cache())
print("adjudicaciones usables unidas a entidad y proveedor:", a.count())

def pares(df):
    return (df.groupBy("entidad_idx", "proveedor_idx")
              .agg(F.count("*").alias("n_adjudicaciones"),
                   F.round(F.sum(F.coalesce("monto_pen", F.lit(0.0))), 2).alias("monto_total"),
                   F.min("fecha_pub").alias("fecha_primera")))

corte = F.lit(FECHA_CORTE).cast("timestamp_ntz")
train_todo = pares(a.filter(F.col("fecha_pub") <  corte)).cache()
test_todo  = pares(a.filter(F.col("fecha_pub") >= corte)).cache()
print("pares en train:", train_todo.count(), "| pares en test:", test_todo.count())
print(f"adjudicaciones en test: {100*a.filter(F.col('fecha_pub') >= corte).count()/a.count():.1f}%")

adjudicaciones usables unidas a entidad y proveedor: 133589


pares en train: 69112 | pares en test: 20108


adjudicaciones en test: 19.0%


In [7]:
# Universo: proveedores y entidades con historia suficiente en train
provs_ok = (train_todo.groupBy("proveedor_idx").agg(F.countDistinct("entidad_idx").alias("n_ent"))
            .filter(F.col("n_ent") >= MIN_ENT_PROV).select("proveedor_idx"))
ents_ok  = (train_todo.groupBy("entidad_idx").agg(F.countDistinct("proveedor_idx").alias("n_prov"))
            .filter(F.col("n_prov") >= MIN_PROV_ENT).select("entidad_idx"))

def en_universo(df):
    return df.join(F.broadcast(provs_ok), "proveedor_idx").join(F.broadcast(ents_ok), "entidad_idx")

train = en_universo(train_todo).cache()
test  = en_universo(test_todo).drop("fecha_primera").cache()

# Se cuentan sobre el train ya filtrado: un proveedor puede pasar su filtro y quedar con menos entidades al aplicar el de entidades
n_p, n_e = train.select("proveedor_idx").distinct().count(), train.select("entidad_idx").distinct().count()
n_train, n_test = train.count(), test.count()
densidad = n_train / (n_p * n_e)
print(f"universo: {n_p:,} proveedores × {n_e:,} entidades")
print(f"pares en train: {n_train:,} | densidad de la matriz: {100*densidad:.3f}%  (el resto son ceros)")
print(f"pares en test (dentro del universo): {n_test:,}")

universo: 9,833 proveedores × 2,331 entidades
pares en train: 44,205 | densidad de la matriz: 0.193%  (el resto son ceros)
pares en test (dentro del universo): 10,303


In [8]:
nuevos = test.join(train.select("entidad_idx", "proveedor_idx"), ["entidad_idx", "proveedor_idx"], "left_anti")
print("pares de test que la entidad NO tenía en train (lo que debe 'descubrir' un recomendador):", nuevos.count())
print("entidades con al menos un par nuevo en test:", nuevos.select("entidad_idx").distinct().count())

for nombre, df in [("proveedor_dim", proveedor_dim), ("entidad_dim", entidad_dim),
                   ("interacciones_train", train), ("interacciones_test", test)]:
    df.coalesce(1).write.mode("overwrite").parquet(f"{RUTA_P2}/{nombre}")
print("Parquet guardados")

pares de test que la entidad NO tenía en train (lo que debe 'descubrir' un recomendador): 4929


entidades con al menos un par nuevo en test: 1333


Parquet guardados


Las columnas de `interacciones_train` son `entidad_idx`, `proveedor_idx`, `n_adjudicaciones`, `monto_total` y `fecha_primera` (la fecha del primer proceso adjudicado en ese par; sirve para separar una validación dentro de train en el notebook 5). `interacciones_test` trae las mismas sin `fecha_primera`.

## 4. Tareas 5 y 7 — Vectores de características
**Proveedores** (una fila por RUC; los consorcios quedan fuera). Cada característica tiene su definición:

| Columna | Qué es |
|---|---|
| `n_participaciones` | procesos en que presentó oferta (aparece en postores) |
| `tasa_exito` | procesos ganados entre procesos en que participó (0 a 1) |
| `n_adjudicaciones` | veces que se le adjudicó |
| `monto_total` | suma de lo adjudicado, en soles |
| `monto_mediano` | monto típico de una adjudicación, en soles |
| `n_entidades` | entidades distintas que le adjudicaron |
| `n_departamentos` | departamentos distintos de esas entidades |
| `concentracion` | parte de sus adjudicaciones que viene de su entidad principal (1 = todas de la misma entidad) |
| `meses_activo` | meses entre su primera y su última participación |
| `categoria_principal` | `goods`, `services` o `works` más frecuente en sus participaciones (solo para describir; no entra al clustering) |

Un proveedor que solo participó y nunca ganó tiene 0 en lo adjudicado. Es un perfil real ("participa y no gana").

In [9]:
dep = entidad_dim.select("entidad_id", F.col("entidad_departamento").alias("departamento"))

part = (post.filter(F.col("ruc").isNotNull() & (F.length("ruc") != 7))
            .select("release_id", F.col("ruc").alias("ruc_p")).distinct()
            .join(procesos.select("release_id", "fecha_pub", "categoria"), "release_id"))

f_part = part.groupBy("ruc_p").agg(
    F.count("*").alias("n_participaciones"),
    F.round(F.months_between(F.max("fecha_pub"), F.min("fecha_pub")), 1).alias("meses_activo"))

# categoría más frecuente
cat = (part.groupBy("ruc_p", "categoria").count()
       .withColumn("pos", F.row_number().over(Window.partitionBy("ruc_p").orderBy(F.desc("count"), "categoria")))
       .filter("pos = 1").select("ruc_p", F.col("categoria").alias("categoria_principal")))

# procesos ganados entre los que participó
adj_real = (adj.filter(F.col("proveedor_ruc").isNotNull() & (F.length("proveedor_ruc") != 7))
               .select("release_id", F.col("proveedor_ruc").alias("ruc_g")).distinct())
f_gan = (part.join(adj_real, (part.release_id == adj_real.release_id) & (part.ruc_p == adj_real.ruc_g))
             .groupBy("ruc_p").agg(F.count("*").alias("n_ganados")))

# lo adjudicado
a_ruc = a.join(proveedor_dim.select("proveedor_idx", "ruc"), "proveedor_idx").join(dep, "entidad_id")
f_adj = a_ruc.groupBy("ruc").agg(
    F.count("*").alias("n_adjudicaciones"),
    F.round(F.sum(F.coalesce("monto_pen", F.lit(0.0))), 2).alias("monto_total"),
    F.round(F.expr("percentile_approx(monto_pen, 0.5)"), 2).alias("monto_mediano"),
    F.countDistinct("entidad_id").alias("n_entidades"),
    F.countDistinct("departamento").alias("n_departamentos"))
top_ent = (a_ruc.groupBy("ruc", "entidad_id").count().groupBy("ruc")
           .agg((F.max("count") / F.sum("count")).alias("concentracion")))

features_proveedor = (proveedor_dim.select("proveedor_idx", "ruc", "nombre")
    .join(f_part, proveedor_dim.ruc == f_part.ruc_p, "left").drop("ruc_p")
    .join(cat,    proveedor_dim.ruc == cat.ruc_p, "left").drop("ruc_p")
    .join(f_gan,  proveedor_dim.ruc == f_gan.ruc_p, "left").drop("ruc_p")
    .join(f_adj, "ruc", "left").join(top_ent, "ruc", "left")
    .fillna(0, subset=["n_participaciones", "n_ganados", "n_adjudicaciones", "monto_total", "monto_mediano",
                       "n_entidades", "n_departamentos", "concentracion", "meses_activo"])
    .withColumn("tasa_exito", F.when(F.col("n_participaciones") > 0,
                                     F.least(F.lit(1.0), F.col("n_ganados") / F.col("n_participaciones"))).otherwise(0.0))
    .select("proveedor_idx", "ruc", "nombre", "n_participaciones", "tasa_exito", "n_adjudicaciones", "monto_total",
            "monto_mediano", "n_entidades", "n_departamentos", "concentracion", "meses_activo", "categoria_principal"))
features_proveedor.cache()
print("proveedores con vector:", features_proveedor.count())
features_proveedor.filter("n_participaciones >= 5").describe("n_participaciones", "tasa_exito", "n_adjudicaciones", "concentracion").show()

proveedores con vector: 104618


+-------+-----------------+------------------+------------------+------------------+
|summary|n_participaciones|        tasa_exito|  n_adjudicaciones|     concentracion|
+-------+-----------------+------------------+------------------+------------------+
|  count|            43115|             43115|             43115|             43115|
|   mean|52.46677490432564|0.0880876717892632|2.5517105415748578|0.3571213749236447|
| stddev|198.0337233800025|0.1606056815223193|  10.4896844617621|0.4198151518030696|
|    min|                5|               0.0|                 0|               0.0|
|    max|             9943|               1.0|              1427|               1.0|
+-------+-----------------+------------------+------------------+------------------+



**Entidades** (una fila por entidad):

| Columna | Qué es |
|---|---|
| `n_procesos` | procesos que publicó |
| `n_proveedores` | proveedores distintos a los que adjudicó |
| `monto_total` y `monto_mediano` | suma y valor típico de lo adjudicado, en soles |
| `pct_bienes`, `pct_servicios`, `pct_obras` | parte de sus procesos en cada categoría (suman 1) |
| `postores_promedio` | postores promedio por proceso (qué tanta competencia tiene) |
| `concentracion` | parte de sus adjudicaciones que va a su proveedor principal |
| `entidad_tipo`, `departamento` | solo para describir; no entran al clustering |

In [10]:
f_proc = (procesos.groupBy("entidad_id").agg(
    F.count("*").alias("n_procesos"),
    F.round(F.avg((F.col("categoria") == "goods").cast("int")), 3).alias("pct_bienes"),
    F.round(F.avg((F.col("categoria") == "services").cast("int")), 3).alias("pct_servicios"),
    F.round(F.avg((F.col("categoria") == "works").cast("int")), 3).alias("pct_obras"),
    F.round(F.avg("n_postores"), 2).alias("postores_promedio")))

f_adj_e = (a.groupBy("entidad_id").agg(
    F.countDistinct("proveedor_idx").alias("n_proveedores"),
    F.round(F.sum(F.coalesce("monto_pen", F.lit(0.0))), 2).alias("monto_total"),
    F.round(F.expr("percentile_approx(monto_pen, 0.5)"), 2).alias("monto_mediano")))
top_prov = (a.groupBy("entidad_id", "proveedor_idx").count().groupBy("entidad_id")
            .agg((F.max("count") / F.sum("count")).alias("concentracion")))

features_entidad = (entidad_dim.select("entidad_idx", "entidad_id", "nombre", "entidad_tipo", F.col("entidad_departamento").alias("departamento"))
    .join(f_proc, "entidad_id", "left").join(f_adj_e, "entidad_id", "left").join(top_prov, "entidad_id", "left")
    .fillna(0, subset=["n_procesos", "n_proveedores", "monto_total", "monto_mediano", "concentracion", "postores_promedio"])
    .select("entidad_idx", "entidad_id", "nombre", "entidad_tipo", "departamento", "n_procesos", "n_proveedores", "monto_total",
            "monto_mediano", "pct_bienes", "pct_servicios", "pct_obras", "postores_promedio", "concentracion"))
features_entidad.cache()
print("entidades con vector:", features_entidad.count())

for nombre, df in [("features_proveedor", features_proveedor), ("features_entidad", features_entidad)]:
    df.coalesce(1).write.mode("overwrite").parquet(f"{RUTA_P2}/{nombre}")
print("Parquet guardados")

entidades con vector: 2992


Parquet guardados


## 5. Tarea 4 — Particionamiento y plan de ejecución
Los Parquet de procesos están **particionados por `anio_pub`** (una carpeta por año). Spark reparte el trabajo en *particiones*; cuando una agregación necesita juntar todas las filas de la misma clave (`groupBy`, `join`), mueve datos entre particiones: eso es un **shuffle** y aparece como `Exchange hashpartitioning` en el plan.

Dos cosas que se miran abajo:
1. **El plan** (`explain()`) de la agregación por par entidad×proveedor.
2. **Cuántas particiones usar** después de un shuffle (`spark.sql.shuffle.partitions`). Desactivamos AQE (que las junta solo) para que se vea el efecto.

In [11]:
print("Particiones al leer procesos:", spark.read.parquet(f"{RUTA_PARQUET}/procesos").rdd.getNumPartitions(),
      "| adjudicaciones:", spark.read.parquet(f"{RUTA_PARQUET}/adjudicaciones").rdd.getNumPartitions())

consulta = a.groupBy("entidad_idx", "proveedor_idx").agg(F.count("*").alias("n"), F.sum("monto_pen").alias("monto"))
buf = StringIO()
with redirect_stdout(buf):
    consulta.explain()
plan = buf.getvalue()
print(plan)

Particiones al leer procesos: 3 | adjudicaciones: 2
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[entidad_idx#636, proveedor_idx#337], functions=[count(1), sum(monto_pen#107)])
   +- Exchange hashpartitioning(entidad_idx#636, proveedor_idx#337, 16), ENSURE_REQUIREMENTS, [plan_id=9538]
      +- HashAggregate(keys=[entidad_idx#636, proveedor_idx#337], functions=[partial_count(1), partial_sum(monto_pen#107)])
         +- InMemoryTableScan [entidad_idx#636, proveedor_idx#337, monto_pen#107]
               +- InMemoryRelation [entidad_idx#636, entidad_id#0, proveedor_idx#337, release_id#100, fecha_pub#11, monto_pen#107], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- AdaptiveSparkPlan isFinalPlan=true
                        +- == Final Plan ==
                           *(4) Project [entidad_idx#636, entidad_id#0, proveedor_idx#337, release_id#100, fecha_pub#11, monto_pen#107]
                           +- *(4) BroadcastHashJoi

In [12]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

def tiempo(n_part):
    spark.conf.set("spark.sql.shuffle.partitions", str(n_part))
    t = []
    for _ in range(3):
        ini = time.perf_counter()
        a.groupBy("entidad_idx", "proveedor_idx").agg(F.count("*"), F.sum("monto_pen")).count()
        t.append(time.perf_counter() - ini)
    return min(t)

tiempos = [{"particiones_shuffle": n, "segundos": round(tiempo(n), 2)} for n in [1, 2, 4, 8, 16, 64, 200]]
spark.conf.set("spark.sql.adaptive.enabled", "true"); spark.conf.set("spark.sql.shuffle.partitions", "16")
pd.DataFrame(tiempos)

,particiones_shuffle,segundos
0,1,0.12
1,2,0.13
2,4,0.14
3,8,0.17
4,16,0.21
5,64,0.44
6,200,0.93


### Lectura
- El plan tiene un **`Exchange hashpartitioning(entidad_idx, proveedor_idx)`**: es el shuffle que junta las filas del mismo par. Antes hay una agregación parcial en cada partición y después la final.
- Pocas particiones usan pocos núcleos; demasiadas (como las 200 por defecto de Spark) **cuestan más en organizar tareas diminutas que en trabajar**. Con datos de este tamaño y una máquina local, 4 a 16 es lo razonable.
- Spark aquí sirve por los 2.5 millones de postores y las uniones. Una vez agregadas, las tablas de proveedores y entidades caben en memoria y los notebooks siguientes trabajan en pandas.

## 6. Reporte para la página web

In [13]:
n_pre = proveedor_dim.count()
reporte = {
    "parametros": {"fecha_corte": FECHA_CORTE, "min_entidades_por_proveedor": MIN_ENT_PROV, "min_proveedores_por_entidad": MIN_PROV_ENT},
    "embudo_adjudicaciones": embudo,
    "proveedores_con_ruc_real": n_pre,
    "ruc_con_varios_nombres": proveedor_dim.filter("n_variantes > 1").count(),
    "postores_filas": n_post, "postores_consorcio_filas": post_consorcio,
    "universo": {"proveedores": n_p, "entidades": n_e, "pares_train": n_train, "pares_test": n_test,
                 "densidad_pct": round(100 * densidad, 4), "pares_test_nuevos": nuevos.count(),
                 "entidades_con_par_nuevo": nuevos.select("entidad_idx").distinct().count()},
    "pct_adjudicaciones_test": round(100 * a.filter(F.col("fecha_pub") >= corte).count() / a.count(), 1),
    "particionamiento": {"plan": plan, "tiempos": tiempos},
}
guardar_json("reporte_base.json", reporte)
reporte["universo"]

guardado: reporte_base.json


{'proveedores': 9833,
 'entidades': 2331,
 'pares_train': 44205,
 'pares_test': 10303,
 'densidad_pct': 0.1929,
 'pares_test_nuevos': 4929,
 'entidades_con_par_nuevo': 1333}

In [14]:
spark.stop()